# MNIST: SVD Optimizer Analysis

Analysis of SVD optimizer (with RMSprop) vs standard optimizers on MNIST classification.
Includes analysis of the `variable_k` setting.

## 1. Setup & Data Loading

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from style import set_style, lr_labels
set_style()

PLOT_DIR = Path('plots/mnist_labelRegression')
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOT_DIR.resolve()}")

In [ ]:
# Load JSONL results
from style import load_results

def sel_fn(filename):
    return "mseed47612" in filename

df = load_results("mnist_scan_labelRegression", selection_fn=sel_fn, slim=False)
print(f"Total: {len(df)} experiment runs")
print(f"Optimizers: {sorted(df['optimizer'].unique())}")
print(f"Batch sizes: {sorted(df['batch_size'].unique())}")

In [ ]:
# Remove incomplete runs from LBFGS
def is_bad(row):
    if row['optimizer'] == 'LBFGS' and np.isnan(row['losses']['val'][-1]):
        return True
    return False
df['is_bad'] = df.apply(is_bad, axis=1)
df = df[~df['is_bad']]
print(f"After removing bad runs: {len(df)} experiment runs")

In [ ]:
# TEMPORARY: remove all but one model seed
print("Selecting one model seed")
df = df[df['model_seed'] == 47612]

In [ ]:
# Helper functions
def get_final_loss(row, loss_type='val'):
    losses = row['losses'][loss_type]
    # Return last non-NaN value (handles instability)
    for val in reversed(losses):
        if val is not None and not (isinstance(val, float) and np.isnan(val)):
            return val
    return np.nan

def get_loss_curve(row, loss_type='val'):
    return np.array(row['losses'][loss_type])

def get_acc_curve(row, acc_type='val_acc'):
    return np.array(row['losses'].get(acc_type, []))

def sliding_average(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')

# Add derived columns once on the full df
df['final_val_loss'] = df.apply(lambda r: get_final_loss(r, 'val'), axis=1)
df['final_train_loss'] = df.apply(lambda r: get_final_loss(r, 'train'), axis=1)
df['final_val_acc'] = df.apply(lambda r: r['losses'].get('val_acc', [np.nan])[-1], axis=1)
df['final_train_acc'] = df.apply(lambda r: r['losses'].get('train_acc', [np.nan])[-1], axis=1)
df['total_time'] = df['losses'].apply(lambda l: l.get('total_time', np.nan))
df['avg_epoch_time'] = df['losses'].apply(lambda l: l.get('avg_epoch_time', np.nan))
df['avg_batch_time_train'] = df['losses'].apply(lambda l: l.get('avg_batch_time_train', np.nan))

def _time_excl_first(row):
    et = row['losses'].get('epoch_times', [])
    return sum(et[1:]) if len(et) > 1 else np.nan

def _avg_batch_excl_first(row):
    bt = row['losses'].get('batch_times_train', [])
    return np.mean(bt[1:]) if len(bt) > 1 else np.nan

df['time_excl_first_epoch'] = df.apply(_time_excl_first, axis=1)
df['avg_batch_time_excl_first'] = df.apply(_avg_batch_excl_first, axis=1)

batch_sizes = sorted(df['batch_size'].unique())
print(f"Available batch sizes: {batch_sizes}")

df_svd = df[(df['optimizer'] == 'SVD') & (df['lr'].str.lower() != 'polyak')].copy()
df_svd_polyak = df[(df['optimizer'] == 'SVD') & (df['lr'].str.lower() == 'polyak')].copy()
df_baseline = df[df['optimizer'] != 'SVD'].copy()

bs = sorted(df['batch_size'].unique())[0] 
batch_sizes = sorted(df['batch_size'].unique())
baseline_optimizers = sorted(df_baseline['optimizer'].unique().tolist())
k_fractions = sorted(df_svd['k_fraction'].dropna().unique())
svd_lrs = sorted(df_svd['lr'].unique())
svd_rtols = sorted(df_svd['rtol'].dropna().unique())

print(f"\nSVD: {len(df_svd)} runs")
print(f"  k_fractions: {k_fractions}")
print(f"  lrs: {svd_lrs}")
print(f"  rtols: {svd_rtols}")
print(f"\nBaseline: {len(df_baseline)} runs")
print(f"  Optimizers: {baseline_optimizers}")
print(f"  lrs: {sorted([l for l in df_baseline['lr'].unique() if l is not None])}")

In [ ]:
from types import SimpleNamespace

def build_context(df, bs):
    """Filter data and compute config variables for one batch size."""
    ctx = SimpleNamespace()
    ctx.bs = bs

    df_bs = df[df['batch_size'] == bs].copy()
    ctx.df = df_bs
    ctx.df_svd = df_bs[df_bs['optimizer'] == 'SVD'].copy()
    ctx.df_baseline = df_bs[df_bs['optimizer'] != 'SVD'].copy()
    ctx.df_svd_fixed_k = ctx.df_svd[ctx.df_svd['variable_k'] == False].copy()
    ctx.df_svd_var_k   = ctx.df_svd[ctx.df_svd['variable_k'] == True].copy()

    ctx.baseline_optimizers = sorted(ctx.df_baseline['optimizer'].unique().tolist())
    ctx.k_fractions = sorted(ctx.df_svd['k_fraction'].dropna().unique())
    ctx.svd_lrs     = sorted(ctx.df_svd['lr'].unique())
    ctx.svd_rtols   = sorted(ctx.df_svd['rtol'].dropna().unique())

    ctx.PLOT_DIR = PLOT_DIR / f'bs{bs}'
    ctx.PLOT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Batch size: {bs}")
    print(f"{'='*60}")
    print(f"SVD total: {len(ctx.df_svd)} runs")
    print(f"  fixed_k: {len(ctx.df_svd_fixed_k)},  variable_k: {len(ctx.df_svd_var_k)}")
    print(f"  k_fractions: {ctx.k_fractions}")
    print(f"  lrs: {ctx.svd_lrs}")
    print(f"  rtols: {ctx.svd_rtols}")
    print(f"Baseline: {len(ctx.df_baseline)} runs  –  {ctx.baseline_optimizers}")
    print(f"Plots → {ctx.PLOT_DIR.resolve()}")
    return ctx


# Custom plots

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'train')
n_epochs = len(train_curve)
epochs_train = np.arange(1,n_epochs + 1)

ax.plot(epochs_train, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(['SGD','PolyakSGD','RMSprop','Adam','LBFGS']):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    #label = f"{opt} ($\eta={lr_labels[best_row['lr']]}$)" if opt != "PolyakSGD" else f"{opt}"
    label = f"{opt}"
    ax.plot(epochs_train, get_loss_curve(best_row, 'train'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Epoch',fontsize=20)
ax.set_ylabel('Train Loss',fontsize=20)
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1,fontsize=16)
ax.set_title(f"MNIST ($B = {bs}$)",fontsize=20)
plt.ylim([2.5e-2,0.3])
plt.xlim(0,n_epochs+1)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7,zorder=0)

plt.tight_layout()
#plt.savefig(PLOT_DIR / 'train_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'val')
n_epochs = len(train_curve)
epochs_train = np.arange(n_epochs)

ax.plot(epochs_train, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(['SGD','PolyakSGD','RMSprop','Adam','LBFGS']):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    #label=f"{opt} ($\eta={lr_labels[best_row['lr']]}$)" if opt != "PolyakSGD" else f"{opt}"
    label= f"{opt}"
    ax.plot(epochs_train, get_loss_curve(best_row, 'val'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"MNIST ($B = {bs}$)")
plt.ylim([None,1])
plt.xlim(0,n_epochs)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7,zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'val_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'train')
wall_times = get_loss_curve(best_svd_row, 'epoch_times')
t = np.cumsum(wall_times)

ax.plot(t, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(['SGD','PolyakSGD','RMSprop','Adam','LBFGS']):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    t = get_loss_curve(best_row, 'epoch_times')
    t = np.cumsum(t)
    #label=f"{opt} ($\eta={lr_labels[best_row['lr']]}$)"
    label = f"{opt}"
    ax.plot(t, get_loss_curve(best_row, 'train'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Wall Time (s)')
ax.set_ylabel('Train Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"MNIST ($B = {bs}$)")
plt.ylim([None,0.4])
#plt.xlim([1,None])
plt.xscale('log')
plt.grid(axis='both', linestyle='--', alpha=0.7,zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_train_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'val')
wall_times = get_loss_curve(best_svd_row, 'epoch_times')
wall_times = np.concatenate(([1], wall_times))  # Add 0 at start to align with epochs
t = np.cumsum(wall_times)

ax.plot(t, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(['SGD','PolyakSGD','RMSprop','Adam','LBFGS']):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    t = get_loss_curve(best_row, 'epoch_times')
    t = np.concatenate(([1], t))  # Add 0 at start to align with epochs
    t = np.cumsum(t)
    #label=f"{opt} ($\eta={lr_labels[best_row['lr']]}$)"
    label = f"{opt}"
    ax.plot(t, get_loss_curve(best_row, 'val'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Wall Time + 1 (sec)')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"MNIST ($B = {bs}$)")
plt.ylim([None,2])
#plt.yticks([1e-1])
plt.xlim([1,None])
plt.xscale('log')
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_val_loss_best.pdf')
plt.show()

In [ ]:
for LR in [0.05,0.1,0.5]:
    for RTOL in [1e-4,1e-3,1e-2,1e-1]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            train_curve = get_loss_curve(row, 'train')
            epochs = np.arange(1, len(train_curve) + 1)
            ax.plot(epochs, train_curve, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        best_sgd_row = df.loc[df_sgd['final_val_loss'].idxmin()]
        ax.plot(epochs, get_loss_curve(best_sgd_row, 'train'), label=f"SGD ($\eta={lr_labels[best_sgd_row['lr']]}$)", color='k', linestyle='--', linewidth=3)
    
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Train Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, MNIST ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.ylim([None,2])
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"train_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

In [ ]:
for LR in [0.05,0.1,0.5]:
    for RTOL in [1e-4,1e-3,1e-2,1e-1]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            val_curve = get_loss_curve(row, 'val')
            epochs = np.arange(0, len(val_curve))
            ax.plot(epochs, val_curve, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        best_sgd_row = df.loc[df_sgd['final_val_loss'].idxmin()]
        ax.plot(epochs, get_loss_curve(best_sgd_row, 'val'), label=f"SGD ($\eta={lr_labels[best_sgd_row['lr']]}$)", color='k', linestyle='--', linewidth=3)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, MNIST ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.ylim([None,2])
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"val_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

In [ ]:
for LR in [0.05,0.1,0.5]:
    for RTOL in [1e-4,1e-3,1e-2,1e-1]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            nonzero_k = np.array(row['svd_info']['num_nonzero_svs'])
            n_epoch = len(get_loss_curve(row, 'train'))
            nonzero_k = nonzero_k.reshape(n_epoch,-1).mean(axis=1)
            epochs = np.arange(1, n_epoch + 1)
            ax.plot(epochs, nonzero_k, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Number of SVs Used by Sven')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, MNIST ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        #plt.savefig(PLOT_DIR / f"train_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

In [ ]:
LR = 0.5
K = 64
colors = sns.color_palette("deep")
fig, ax = plt.subplots(figsize=(8, 6))
for j,K in enumerate([16,32,64]):
    df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['k'] == K)]
    rtol_values = sorted(df_sel['rtol'].unique())
    rtol_styles = {1e-4: 'dashed', 1e-3: 'solid', 1e-2: 'dashdot', 1e-1: 'dotted'}
    for i,rtol in enumerate(rtol_values):
        row = df_sel[df_sel['rtol'] == rtol]
        assert len(row) == 1, f"Expected exactly one run for rtol={rtol}, found {len(row)}"
        row = row.iloc[0]
        val_loss = row['losses']['val']
        epochs = np.arange(0, len(val_loss))
        ax.plot(epochs, val_loss, color=colors[j], linewidth=3, linestyle=rtol_styles[rtol])

rtol_lines = [Line2D([0], [0], color='k', linestyle=rtol_styles[rtol], linewidth=3) for rtol in rtol_values]
rtol_labels_legend = [f"${lr_labels[rtol]}$" for rtol in rtol_values]
custom_lines = [Line2D([0], [0], color=colors[j], linewidth=3) for j in range(3)]
custom_labels = [f"$k={k}$" for k in [16,32,64]]
leg1 = plt.legend(custom_lines, custom_labels, loc='upper right', ncol=1, frameon=True, framealpha=1, bbox_to_anchor=(0.84, 0.945))
plt.gca().add_artist(leg1)
leg2 = plt.legend(rtol_lines, rtol_labels_legend, loc='upper right', frameon=True, framealpha=1, title='rtol')
plt.gca().add_artist(leg2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_title(f"Sven, MNIST ($B = {bs}$, $\eta={LR}$)")
plt.xlim(0,n_epochs+1)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.yscale('log')
plt.ylim([None,0.5])

plt.tight_layout()
plt.savefig(PLOT_DIR / f"val_loss_overlay_rtol_and_k.pdf")
plt.show()

## Section functions

### 2. Best Performance Comparison

In [ ]:
def plot_best_performance(ctx):
    """Bar charts comparing best final val loss and accuracy per optimizer."""
    best_runs = []

    if len(ctx.df_svd_fixed_k) > 0:
        best = ctx.df_svd_fixed_k.loc[ctx.df_svd_fixed_k['final_val_loss'].idxmin()]
        best_runs.append({
            'optimizer': 'SVD (fixed k)',
            'final_val_loss': best['final_val_loss'],
            'final_val_acc': best['final_val_acc'],
            'lr': best['lr'], 'k': best['k'],
            'k_fraction': best['k_fraction'], 'rtol': best['rtol'],
            'total_time': best['total_time'],
        })

    if len(ctx.df_svd_var_k) > 0:
        best = ctx.df_svd_var_k.loc[ctx.df_svd_var_k['final_val_loss'].idxmin()]
        best_runs.append({
            'optimizer': 'SVD (var k)',
            'final_val_loss': best['final_val_loss'],
            'final_val_acc': best['final_val_acc'],
            'lr': best['lr'], 'k': best['k'],
            'k_fraction': best['k_fraction'], 'rtol': best['rtol'],
            'total_time': best['total_time'],
        })

    for opt in ctx.baseline_optimizers:
        opt_df = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt]
        best = opt_df.loc[opt_df['final_val_loss'].idxmin()]
        best_runs.append({
            'optimizer': opt,
            'final_val_loss': best['final_val_loss'],
            'final_val_acc': best['final_val_acc'],
            'lr': best['lr'], 'k': None, 'k_fraction': None,
            'rtol': None, 'total_time': best['total_time'],
        })

    ctx.best_df = pd.DataFrame(best_runs)
    print(f"Best performance for each optimizer (bs={ctx.bs}):")
    print(ctx.best_df.to_string(index=False))

    n = len(ctx.best_df)
    colors = [f'C{i}' for i in range(n)]
    x = np.arange(n)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    bars = ax.bar(x, ctx.best_df['final_val_loss'], color=colors)
    ax.set_xticks(x); ax.set_xticklabels(ctx.best_df['optimizer'], rotation=20, ha='right')
    ax.set_ylabel('Final Validation Loss')
    ax.set_title(f'Best Final Validation Loss (B={ctx.bs})')
    ax.set_yscale('log')
    for bar, val in zip(bars, ctx.best_df['final_val_loss']):
        ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.3f}',
                ha='center', va='bottom', fontsize=9)

    ax = axes[1]
    bars = ax.bar(x, ctx.best_df['final_val_acc'] * 100, color=colors)
    ax.set_xticks(x); ax.set_xticklabels(ctx.best_df['optimizer'], rotation=20, ha='right')
    ax.set_ylabel('Final Validation Accuracy (%)')
    ax.set_title(f'Best Final Validation Accuracy (B={ctx.bs})')
    for bar, val in zip(bars, ctx.best_df['final_val_acc']):
        ax.text(bar.get_x() + bar.get_width()/2, val*100, f'{val*100:.1f}%',
                ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'best_optimizer_comparison.pdf')
    plt.show()


### 3. Training Curves: Best Configurations

In [ ]:
def _collect_best_rows(ctx, criterion='final_val_loss', minimize=True):
    """Collect (name, row, label) for best config of each optimizer by criterion."""
    best_rows = []
    if len(ctx.df_svd_fixed_k) > 0:
        idx = (ctx.df_svd_fixed_k[criterion].idxmin() if minimize
               else ctx.df_svd_fixed_k[criterion].idxmax())
        row = ctx.df.loc[idx]
        best_rows.append(('SVD fixed-k', row,
                          f"SVD fixed-k (lr={row['lr']}, k={int(row['k'])})"))
    if len(ctx.df_svd_var_k) > 0:
        idx = (ctx.df_svd_var_k[criterion].idxmin() if minimize
               else ctx.df_svd_var_k[criterion].idxmax())
        row = ctx.df.loc[idx]
        best_rows.append(('SVD var-k', row,
                          f"SVD var-k (lr={row['lr']}, k={int(row['k'])})"))
    for opt in ctx.baseline_optimizers:
        opt_df = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt]
        idx = (opt_df[criterion].idxmin() if minimize else opt_df[criterion].idxmax())
        row = ctx.df.loc[idx]
        best_rows.append((opt, row, f"{opt} (lr={row['lr']:.0e})"))
    return best_rows


def _plot_loss_acc_curves(best_rows, title_suffix, fname, ctx):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for i, (name, row, label) in enumerate(best_rows):
        train_curve = get_loss_curve(row, 'train')
        val_curve   = get_loss_curve(row, 'val')
        axes[0].plot(range(1, len(train_curve)+1), train_curve, f'C{i}-')
        axes[0].plot(range(len(val_curve)), val_curve, f'C{i}--')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_yscale('log')
    axes[0].set_title(f'MNIST, Best Runs {title_suffix} ($B={ctx.bs}$)')
    leg1 = axes[0].legend(handles=[Line2D([], [], color=f'C{i}', linestyle='-', label=best_rows[i][2])
                                    for i in range(len(best_rows))], loc='upper right', fontsize=10)
    axes[0].add_artist(leg1)
    axes[0].legend(handles=[Line2D([], [], color='k', linestyle='-', label='Train'),
                             Line2D([], [], color='k', linestyle='--', label='Val')], loc='lower left')

    for i, (name, row, label) in enumerate(best_rows):
        ta = get_acc_curve(row, 'train_acc'); va = get_acc_curve(row, 'val_acc')
        if len(ta) > 0: axes[1].plot(range(1, len(ta)+1), ta*100, f'C{i}-')
        if len(va) > 0: axes[1].plot(range(len(va)), va*100, f'C{i}--')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_ylim(80, 100)
    axes[1].set_title(f'MNIST, Best Runs {title_suffix} ($B={ctx.bs}$)')
    leg1 = axes[1].legend(handles=[Line2D([], [], color=f'C{i}', linestyle='-', label=best_rows[i][2])
                                    for i in range(len(best_rows))], loc='lower right', fontsize=10)
    axes[1].add_artist(leg1)
    axes[1].legend(handles=[Line2D([], [], color='k', linestyle='-', label='Train'),
                             Line2D([], [], color='k', linestyle='--', label='Val')], loc='upper left')
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / fname)
    plt.show()


def plot_training_curves(ctx):
    """Loss/accuracy training curves for best configs (by val loss, train loss, val acc)."""
    # Best by final val loss
    best_rows = _collect_best_rows(ctx, 'final_val_loss', minimize=True)
    ctx.best_rows = best_rows  # stored for later sections
    _plot_loss_acc_curves(best_rows, 'by Final Val Loss',
                          'train_val_curves_best_byFinalValLoss.pdf', ctx)

    # Best by final train loss
    br = _collect_best_rows(ctx, 'final_train_loss', minimize=True)
    _plot_loss_acc_curves(br, 'by Final Train Loss',
                          'train_val_curves_best_byFinalTrainLoss.pdf', ctx)

    # Best by final val accuracy
    br = _collect_best_rows(ctx, 'final_val_acc', minimize=False)
    _plot_loss_acc_curves(br, 'by Final Val Accuracy',
                          'train_val_curves_best_byFinalValAcc.pdf', ctx)

    # Best SVD at each LR vs best baselines
    baseline_styles = {'Adam': ('C1', 's-'), 'RMSprop': ('C3', 'D-'), 'SGD': ('C4', '^-')}
    best_svd_per_lr = []
    for lr in ctx.svd_lrs:
        lr_runs = ctx.df_svd[ctx.df_svd['lr'] == lr]
        if len(lr_runs) == 0: continue
        best = ctx.df.loc[lr_runs['final_val_loss'].idxmin()]
        vk = best.get('variable_k', False)
        best_svd_per_lr.append({
            'lr': lr, 'row': best,
            'label': f"SVD lr={lr}" + (" (var-k)" if vk else ""),
            'final_val_loss': best['final_val_loss'],
            'final_val_acc': best['final_val_acc'],
            'k_fraction': best['k_fraction'], 'rtol': best['rtol'], 'variable_k': vk,
        })

    print(f"\nBest SVD config at each LR (bs={ctx.bs}):")
    print(f"{'LR':>8s}  {'Val Loss':>10s}  {'Val Acc':>8s}  {'k':>6s}  {'rtol':>10s}  {'var_k':>6s}")
    for e in best_svd_per_lr:
        print(f"{e['lr']:8g}  {e['final_val_loss']:10.4f}  {e['final_val_acc']*100:7.2f}%  "
              f"{e['k_fraction']:6.4f}  {e['rtol']:10.1e}  {str(e['variable_k']):>6s}")

    n_svd = len(best_svd_per_lr)
    cmap = plt.cm.viridis
    svd_colors = [cmap(i / max(n_svd - 1, 1)) for i in range(n_svd)]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    ax = axes[0]
    for i, entry in enumerate(best_svd_per_lr):
        row = entry['row']
        val_curve = get_loss_curve(row, 'val')
        ax.plot(range(len(val_curve)), val_curve, color=svd_colors[i], lw=2,
                alpha=0.8, label=entry['label'])
    for opt in ctx.baseline_optimizers:
        opt_df = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt]
        row = ctx.df.loc[opt_df['final_val_loss'].idxmin()]
        val_curve = get_loss_curve(row, 'val')
        color, style = baseline_styles.get(opt, ('C5', 'o-'))
        ax.plot(range(len(val_curve)), val_curve, style, color=color, lw=2.5,
                label=f"{opt} (lr={row['lr']:.0e})", markersize=4,
                markevery=max(1, len(val_curve)//10))
    ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Loss'); ax.set_yscale('log')
    ax.set_title(f'Best SVD at Each LR vs Best Baselines: Val Loss (B={ctx.bs})')
    ax.legend(fontsize=9, loc='upper right', ncol=1)

    ax = axes[1]
    for i, entry in enumerate(best_svd_per_lr):
        row = entry['row']
        val_acc = get_acc_curve(row, 'val_acc')
        if len(val_acc) > 0:
            ax.plot(range(1, len(val_acc)+1), val_acc*100, color=svd_colors[i],
                    lw=2, alpha=0.8, label=entry['label'])
    for opt in ctx.baseline_optimizers:
        opt_df = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt]
        row = ctx.df.loc[opt_df['final_val_loss'].idxmin()]
        val_acc = get_acc_curve(row, 'val_acc')
        color, style = baseline_styles.get(opt, ('C5', 'o-'))
        if len(val_acc) > 0:
            ax.plot(range(1, len(val_acc)+1), val_acc*100, style, color=color, lw=2.5,
                    label=f"{opt} (lr={row['lr']:.0e})", markersize=4,
                    markevery=max(1, len(val_acc)//10))
    ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Accuracy (%)')
    ax.set_title(f'Best SVD at Each LR vs Best Baselines: Val Accuracy (B={ctx.bs})')
    ax.legend(fontsize=9, loc='lower right', ncol=1)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'best_svd_per_lr_vs_baselines.pdf')
    plt.show()

    # Bar charts: final val loss and accuracy for SVD-per-LR vs baselines
    all_labels  = [e['label'] for e in best_svd_per_lr] + ctx.baseline_optimizers
    all_losses  = [e['final_val_loss'] for e in best_svd_per_lr]
    all_accs    = [e['final_val_acc'] for e in best_svd_per_lr]
    for opt in ctx.baseline_optimizers:
        opt_df = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt]
        best = opt_df.loc[opt_df['final_val_loss'].idxmin()]
        all_losses.append(best['final_val_loss'])
        all_accs.append(best['final_val_acc'])
    bar_colors = list(svd_colors) + [baseline_styles.get(opt, ('C5', 'o-'))[0]
                                     for opt in ctx.baseline_optimizers]
    x = np.arange(len(all_labels))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    ax = axes[0]
    bars = ax.bar(x, all_losses, color=bar_colors)
    ax.set_xticks(x); ax.set_xticklabels(all_labels, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Final Validation Loss')
    ax.set_title(f'Final Val Loss: Best SVD per LR vs Baselines (B={ctx.bs})')
    ax.set_yscale('log')
    for bar, val in zip(bars, all_losses):
        ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.3f}',
                ha='center', va='bottom', fontsize=8, rotation=45)
    ax = axes[1]
    bars = ax.bar(x, [a*100 for a in all_accs], color=bar_colors)
    ax.set_xticks(x); ax.set_xticklabels(all_labels, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Final Validation Accuracy (%)')
    ax.set_title(f'Final Val Accuracy: Best SVD per LR vs Baselines (B={ctx.bs})')
    for bar, val in zip(bars, all_accs):
        ax.text(bar.get_x() + bar.get_width()/2, val*100, f'{val*100:.1f}%',
                ha='center', va='bottom', fontsize=8, rotation=45)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'best_svd_per_lr_vs_baselines_bars.pdf')
    plt.show()

    # Batch-level training loss for best-by-val-loss configs
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, (name, row, label) in enumerate(ctx.best_rows):
        bl = get_loss_curve(row, 'train_batch')
        n_ep = len(row['losses']['train'])
        ex = np.linspace(0, n_ep, len(bl))
        ax.plot(ex, bl, f'C{i}-', alpha=0.5, lw=1)
        sw = max(1, len(bl) // (n_ep * 5))
        if sw > 1:
            bl_smooth = sliding_average(bl, window=sw)
            ax.plot(ex[sw-1:], bl_smooth, f'C{i}-', lw=2, label=label)
        else:
            ax.plot([], [], f'C{i}-', lw=2, label=label)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Training Loss'); ax.set_yscale('log')
    ax.set_title(f'MNIST: Batch-level Training Loss (B={ctx.bs})')
    ax.legend(loc='upper right', fontsize=10)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'training_loss_batchwise_best.pdf')
    plt.show()


### 4. Variable k vs Fixed k Analysis

In [ ]:
def plot_variable_k(ctx):
    """Variable-k vs fixed-k scatter, curves, k_used evolution, and per-config grids."""
    if len(ctx.df_svd_var_k) == 0:
        print(f"No variable-k data for bs={ctx.bs}, skipping section 4.")
        return

    # Matched hyperparameter pairs
    shared_configs = pd.merge(
        ctx.df_svd_fixed_k[['k_fraction', 'lr', 'rtol', 'final_val_loss',
                             'final_val_acc', 'total_time']],
        ctx.df_svd_var_k[['k_fraction', 'lr', 'rtol', 'final_val_loss',
                           'final_val_acc', 'total_time']],
        on=['k_fraction', 'lr'], suffixes=('_fixed', '_var')
    )
    ctx.shared_configs = shared_configs
    shared_configs['loss_ratio'] = (shared_configs['final_val_loss_var'] /
                                    shared_configs['final_val_loss_fixed'])
    shared_configs['acc_diff']   = (shared_configs['final_val_acc_var'] -
                                    shared_configs['final_val_acc_fixed'])
    shared_configs['time_ratio'] = (shared_configs['total_time_var'] /
                                    shared_configs['total_time_fixed'])
    print(f"Matched config pairs: {len(shared_configs)}")
    print(f"  Loss ratio (var/fixed): median={shared_configs['loss_ratio'].median():.3f}")
    print(f"  Acc diff (var-fixed):   median={shared_configs['acc_diff'].median()*100:.2f}%")
    print(f"  Time ratio (var/fixed): median={shared_configs['time_ratio'].median():.3f}")

    # Scatter: fixed vs variable final val loss
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    ax = axes[0]
    ax.scatter(shared_configs['final_val_loss_fixed'], shared_configs['final_val_loss_var'],
               c='C0', alpha=0.7, s=50)
    lims = [min(shared_configs['final_val_loss_fixed'].min(),
                shared_configs['final_val_loss_var'].min()) * 0.8,
            max(shared_configs['final_val_loss_fixed'].max(),
                shared_configs['final_val_loss_var'].max()) * 1.2]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
    ax.set_xlabel('Final Val Loss (fixed k)'); ax.set_ylabel('Final Val Loss (variable k)')
    ax.set_title(f'Variable k vs Fixed k: Validation Loss (B={ctx.bs})'); ax.legend()
    ax = axes[1]
    ax.scatter(shared_configs['final_val_acc_fixed']*100,
               shared_configs['final_val_acc_var']*100, c='C0', alpha=0.7, s=50)
    lims = [min(shared_configs['final_val_acc_fixed'].min(),
                shared_configs['final_val_acc_var'].min())*100 - 1,
            max(shared_configs['final_val_acc_fixed'].max(),
                shared_configs['final_val_acc_var'].max())*100 + 1]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
    ax.set_xlabel('Final Val Acc (%) (fixed k)'); ax.set_ylabel('Final Val Acc (%) (variable k)')
    ax.set_title(f'Variable k vs Fixed k: Validation Accuracy (B={ctx.bs})'); ax.legend()
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'variable_k_vs_fixed_k_scatter.pdf')
    plt.show()

    # Training curves: best fixed vs best variable
    best_fixed = ctx.df_svd_fixed_k.loc[ctx.df_svd_fixed_k['final_val_loss'].idxmin()]
    best_var   = ctx.df_svd_var_k.loc[ctx.df_svd_var_k['final_val_loss'].idxmin()]
    matched_var = ctx.df_svd_var_k[
        (ctx.df_svd_var_k['k_fraction'] == best_fixed['k_fraction']) &
        (ctx.df_svd_var_k['lr'] == best_fixed['lr']) &
        (ctx.df_svd_var_k['rtol'] == best_fixed['rtol'])]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    ax = axes[0]
    ax.plot(range(1, len(best_fixed['losses']['train'])+1),
            get_loss_curve(best_fixed, 'train'), 'C0-',
            label=f"Fixed k (lr={best_fixed['lr']}, k={int(best_fixed['k'])})")
    ax.plot(range(len(best_fixed['losses']['val'])), get_loss_curve(best_fixed, 'val'), 'C0--')
    ax.plot(range(1, len(best_var['losses']['train'])+1),
            get_loss_curve(best_var, 'train'), 'C1-',
            label=f"Var k (lr={best_var['lr']}, k={int(best_var['k'])})")
    ax.plot(range(len(best_var['losses']['val'])), get_loss_curve(best_var, 'val'), 'C1--')
    if len(matched_var) > 0:
        mv = matched_var.iloc[0]
        ax.plot(range(1, len(mv['losses']['train'])+1), get_loss_curve(mv, 'train'), 'C2-',
                label=f"Var k matched (lr={mv['lr']}, k={int(mv['k'])})")
        ax.plot(range(len(mv['losses']['val'])), get_loss_curve(mv, 'val'), 'C2--')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_yscale('log')
    ax.set_title(f'Fixed k vs Variable k: Loss (B={ctx.bs})')
    leg1 = ax.legend(loc='upper right', fontsize=10); ax.add_artist(leg1)
    ax.legend(handles=[Line2D([], [], color='k', linestyle='-', label='Train'),
                        Line2D([], [], color='k', linestyle='--', label='Val')], loc='lower left')
    ax = axes[1]
    for (name, row, col) in [('Fixed k', best_fixed, 'C0'), ('Var k', best_var, 'C1')]:
        ta = get_acc_curve(row, 'train_acc'); va = get_acc_curve(row, 'val_acc')
        if len(ta) > 0:
            ax.plot(range(1, len(ta)+1), ta*100, f'{col}-',
                    label=f"{name} (lr={row['lr']}, k={int(row['k'])})")
            ax.plot(range(1, len(va)+1), va*100, f'{col}--')
    if len(matched_var) > 0:
        mv = matched_var.iloc[0]
        ta = get_acc_curve(mv, 'train_acc'); va = get_acc_curve(mv, 'val_acc')
        if len(ta) > 0:
            ax.plot(range(1, len(ta)+1), ta*100, 'C2-',
                    label=f"Var k matched (lr={mv['lr']}, k={int(mv['k'])})")
            ax.plot(range(1, len(va)+1), va*100, 'C2--')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.set_title(f'Fixed k vs Variable k: Accuracy (B={ctx.bs})')
    leg1 = ax.legend(loc='lower right', fontsize=10); ax.add_artist(leg1)
    ax.legend(handles=[Line2D([], [], color='k', linestyle='-', label='Train'),
                        Line2D([], [], color='k', linestyle='--', label='Val')], loc='upper left')
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'variable_k_vs_fixed_k_curves.pdf')
    plt.show()

    # k_used evolution (best variable-k)
    k_used = best_var['svd_info']['k_used']
    if len(k_used) > 0:
        fig, ax = plt.subplots(figsize=(8, 6))
        n_ep = len(best_var['losses']['train'])
        x = np.linspace(0, n_ep, len(k_used))
        sw = max(1, len(k_used) // (n_ep * 5))
        ax.plot(x, k_used, 'C0-', alpha=0.3, lw=1)
        if sw > 1:
            ax.plot(x[sw-1:], sliding_average(k_used, window=sw), 'C0-', lw=2)
        ax.axhline(y=best_var['k'], color='r', linestyle='--', alpha=0.5,
                   label=f'Max k = {int(best_var["k"])}')
        ax.set_xlabel('Epoch'); ax.set_ylabel('k used')
        ax.set_title(f"Adaptive k (lr={best_var['lr']}, k_max={int(best_var['k'])}, "
                     f"rtol={best_var['rtol']}, B={ctx.bs})")
        ax.legend()
        plt.tight_layout()
        plt.savefig(ctx.PLOT_DIR / 'variable_k_evolution_best.pdf')
        plt.show()

    # k_used evolution by k_fraction
    fig, ax = plt.subplots(figsize=(8, 6))
    legend_entries = []
    for i, kf in enumerate(ctx.k_fractions):
        kf_data = ctx.df_svd_var_k[ctx.df_svd_var_k['k_fraction'] == kf]
        if len(kf_data) == 0: continue
        best = kf_data.loc[kf_data['final_val_loss'].idxmin()]
        k_used = best['svd_info']['k_used']
        if len(k_used) == 0: continue
        n_ep = len(best['losses']['train'])
        x = np.linspace(0, n_ep, len(k_used))
        sw = max(1, len(k_used) // (n_ep * 5))
        y = sliding_average(k_used, window=sw) if sw > 1 else k_used
        xs = x[sw-1:] if sw > 1 else x
        ax.plot(xs, y, f'C{i}-', lw=2)
        legend_entries.append(Line2D([], [], color=f'C{i}',
                                     label=f'k={int(kf*ctx.bs)}'))
    ax.set_xlabel('Epoch'); ax.set_ylabel('k used')
    ax.set_title(f'Adaptive k Evolution by k_fraction (B={ctx.bs})')
    if legend_entries: ax.legend(handles=legend_entries)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'variable_k_evolution_by_kfraction.pdf')
    plt.show()

    # Wall time scatter: variable vs fixed
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(shared_configs['total_time_fixed'], shared_configs['total_time_var'],
               c='C0', alpha=0.7, s=50)
    lims = [min(shared_configs['total_time_fixed'].min(),
                shared_configs['total_time_var'].min()) * 0.8,
            max(shared_configs['total_time_fixed'].max(),
                shared_configs['total_time_var'].max()) * 1.2]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
    ax.set_xlabel('Total Time (s) - Fixed k'); ax.set_ylabel('Total Time (s) - Variable k')
    ax.set_title(f'Variable k vs Fixed k: Wall Time (B={ctx.bs})'); ax.legend()
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'variable_k_vs_fixed_k_walltime.pdf')
    plt.show()

    # Grid plots: val loss / val acc / train loss / train acc per (k_fraction, lr)
    rtol_colors = {rtol: f'C{i}' for i, rtol in enumerate(ctx.svd_rtols)}
    vk_color = 'k'
    legend_handles = [Line2D([], [], color=vk_color, linestyle='--', lw=2, label='variable k')]
    for rtol in ctx.svd_rtols:
        legend_handles.append(Line2D([], [], color=rtol_colors[rtol], lw=1.5,
                                     label=f'fixed k, rtol={rtol:.0e}'))

    for metric, ylabel, fname, acc_mode in [
        ('val',      'Val Loss',    'vark_vs_fixedk_per_config_val_loss.pdf',  False),
        ('val_acc',  'Val Acc (%)', 'vark_vs_fixedk_per_config_val_acc.pdf',   True),
        ('train',    'Train Loss',  'vark_vs_fixedk_per_config_train_loss.pdf', False),
        ('train_acc','Train Acc (%)', 'vark_vs_fixedk_per_config_train_acc.pdf', True),
    ]:
        nrows = len(ctx.k_fractions); ncols = len(ctx.svd_lrs)
        if nrows == 0 or ncols == 0: continue
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(3.2*ncols, 2.8*nrows), sharex=True)
        if nrows == 1: axes = np.array([axes])
        if ncols == 1: axes = axes[:, np.newaxis]
        for ri, kf in enumerate(ctx.k_fractions):
            for ci, lr in enumerate(ctx.svd_lrs):
                ax = axes[ri, ci]
                fixed_runs = ctx.df_svd_fixed_k[
                    (ctx.df_svd_fixed_k['k_fraction'] == kf) & (ctx.df_svd_fixed_k['lr'] == lr)]
                for _, run in fixed_runs.iterrows():
                    rtol = run['rtol']
                    curve = (get_acc_curve(run, metric)*100 if acc_mode
                             else get_loss_curve(run, metric))
                    xvals = (range(1, len(curve)+1) if acc_mode else range(len(curve)))
                    if len(curve) > 0:
                        ax.plot(xvals, curve, color=rtol_colors[rtol], lw=1.5, alpha=0.8)
                var_runs = ctx.df_svd_var_k[
                    (ctx.df_svd_var_k['k_fraction'] == kf) & (ctx.df_svd_var_k['lr'] == lr)]
                for _, run in var_runs.iterrows():
                    curve = (get_acc_curve(run, metric)*100 if acc_mode
                             else get_loss_curve(run, metric))
                    xvals = (range(1, len(curve)+1) if acc_mode else range(len(curve)))
                    if len(curve) > 0:
                        ax.plot(xvals, curve, color=vk_color, lw=2, linestyle='--', alpha=0.9)
                if not acc_mode: ax.set_yscale('log')
                if ri == 0: ax.set_title(f'lr={lr}', fontsize=10)
                if ri == nrows - 1: ax.set_xlabel('Epoch', fontsize=9)
                if ci == 0: ax.set_ylabel(f'k={int(kf*ctx.bs)}\n{ylabel}', fontsize=9)
                ax.tick_params(labelsize=8)
        fig.legend(handles=legend_handles, loc='lower center', ncol=len(ctx.svd_rtols)+1,
                   fontsize=10, bbox_to_anchor=(0.5, -0.02))
        title_map = {'val': 'Validation Loss', 'val_acc': 'Validation Accuracy',
                     'train': 'Train Loss', 'train_acc': 'Train Accuracy'}
        fig.suptitle(f'Variable k vs Fixed k: {title_map[metric]} (B={ctx.bs})',
                     fontsize=14, y=1.01)
        plt.tight_layout()
        plt.savefig(ctx.PLOT_DIR / fname, bbox_inches='tight')
        plt.show()


### 5. SVD Hyperparameter Sensitivity

In [ ]:
def plot_hyperparam_sensitivity(ctx):
    """Heatmaps and sensitivity curves for SVD hyperparameters."""
    # Heatmaps for fixed_k
    for subset, label, fname in [
        (ctx.df_svd_fixed_k, 'fixed k',    'svd_heatmap_fixed_k.pdf'),
        (ctx.df_svd_var_k,   'variable k', 'svd_heatmap_variable_k.pdf'),
    ]:
        if len(subset) == 0: continue
        fig, axes = plt.subplots(1, len(ctx.svd_rtols),
                                 figsize=(4*len(ctx.svd_rtols), 4))
        if len(ctx.svd_rtols) == 1: axes = [axes]
        vmin = np.log10(ctx.df_svd['final_val_loss'].min())
        vmax = np.log10(ctx.df_svd['final_val_loss'].max())
        for ax, rtol in zip(axes, ctx.svd_rtols):
            data = subset[subset['rtol'] == rtol]
            pivot = data.pivot_table(values='final_val_loss', index='k_fraction',
                                     columns='lr', aggfunc='first')
            im = ax.imshow(np.log10(pivot.values), aspect='auto', cmap='viridis',
                           vmin=vmin, vmax=vmax)
            ax.set_xticks(range(len(pivot.columns)))
            ax.set_xticklabels([f'{lr}' for lr in pivot.columns], rotation=45)
            ax.set_yticks(range(len(pivot.index)))
            ax.set_yticklabels([f'{int(kf*ctx.bs)}' for kf in pivot.index])
            ax.set_xlabel('Learning Rate'); ax.set_ylabel('k')
            ax.set_title(f'rtol = {rtol} ({label}, B={ctx.bs})')
            if ax != axes[0]: ax.set_ylabel(''); ax.set_yticklabels([])
        fig.subplots_adjust(right=0.85)
        cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.7])
        cbar = fig.colorbar(im, cax=cbar_ax); cbar.set_label('log10(Validation Loss)')
        plt.savefig(ctx.PLOT_DIR / fname, bbox_inches='tight')
        plt.show()

    # LR sensitivity (best across rtol) for fixed and variable k
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, (subset, title) in zip(axes, [
        (ctx.df_svd_fixed_k, 'Fixed k'), (ctx.df_svd_var_k, 'Variable k')
    ]):
        if len(subset) == 0: continue
        for i, kf in enumerate(ctx.k_fractions):
            kf_data = subset[subset['k_fraction'] == kf]
            if len(kf_data) == 0: continue
            best_per_lr = kf_data.groupby('lr')['final_val_loss'].min().reset_index()
            ax.plot(best_per_lr['lr'], best_per_lr['final_val_loss'], 'o-',
                    label=f'k={int(kf*ctx.bs)}', color=f'C{i}')
        ax.set_xlabel('Learning Rate'); ax.set_ylabel('Final Validation Loss')
        ax.set_title(f'LR Sensitivity – {title} (B={ctx.bs})')
        ax.set_xscale('log'); ax.set_yscale('log'); ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'lr_sensitivity_fixed_vs_var_k.pdf')
    plt.show()

    # k effect at each lr (fixed k)
    for lr in ctx.svd_lrs:
        fig, ax = plt.subplots(figsize=(8, 6))
        for i, rtol in enumerate(ctx.svd_rtols):
            data = ctx.df_svd_fixed_k[
                (ctx.df_svd_fixed_k['lr'] == lr) & (ctx.df_svd_fixed_k['rtol'] == rtol)
            ].sort_values('k_fraction')
            ax.plot(data['k_fraction'] * ctx.bs, data['final_val_loss'], 'o-',
                    label=f'rtol={rtol}', color=f'C{i}')
        ax.set_xlabel('k'); ax.set_ylabel('Final Validation Loss')
        ax.set_title(f'k Effect (lr={lr}, fixed k, B={ctx.bs})')
        ax.set_yscale('log'); ax.legend()
        plt.tight_layout()
        plt.savefig(ctx.PLOT_DIR / f'k_fraction_effect_fixedk_lr{lr}.pdf')
        plt.show()

### 6. Singular Value Analysis

In [ ]:
def plot_singular_values(ctx):
    """Nonzero SV counts over training and SV spectrum evolution."""
    if len(ctx.df_svd_fixed_k) == 0: return

    best_svd_row = ctx.df.loc[ctx.df_svd_fixed_k['final_val_loss'].idxmin()]

    # Nonzero SVs vs training loss (dual axis)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax2 = ax.twinx()
    num_nonzero = best_svd_row['svd_info']['num_nonzero_svs']
    train_batch = best_svd_row['losses']['train_batch']
    n_ep = len(best_svd_row['losses']['train'])
    x = np.linspace(0, n_ep, len(num_nonzero))
    sw = max(1, len(num_nonzero) // (n_ep * 5))
    ax.plot(x[sw-1:], sliding_average(num_nonzero, window=sw), 'C0-', lw=2)
    ax2.plot(x[sw-1:], sliding_average(train_batch, window=sw), 'C1--', lw=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('# Nonzero Singular Values', color='C0')
    ax2.set_ylabel('Training Loss', color='C1'); ax2.set_yscale('log')
    ax.set_title(f"Best SVD (fixed k): lr={best_svd_row['lr']}, "
                 f"k={int(best_svd_row['k'])}, B={ctx.bs}")
    ax.legend(handles=[Line2D([], [], color='C0', linestyle='-', label='# Nonzero SVs'),
                        Line2D([], [], color='C1', linestyle='--', label='Training Loss')],
              loc='upper right')
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'nonzero_svs_vs_trainloss_best_fixedk.pdf')
    plt.show()

    # SV spectrum evolution
    svs_list = best_svd_row['svd_info']['svs']
    if svs_list is not None and len(svs_list) > 0:
        total_batches = len(svs_list)
        bpe = total_batches // n_ep
        sample_points = [
            (0, 'Batch 0 (Start)'),
            (bpe, 'Epoch 1'),
            (3*bpe, 'Epoch 3'),
            (n_ep//2*bpe, f'Epoch {n_ep//2}'),
            (3*n_ep//4*bpe, f'Epoch {3*n_ep//4}'),
            (total_batches - 1, f'Epoch {n_ep} (End)'),
        ]
        fig, axes = plt.subplots(2, 3, figsize=(14, 8))
        for ax, (bidx, title) in zip(axes.flat, sample_points):
            svs = svs_list[min(bidx, len(svs_list)-1)]
            ax.bar(range(len(svs)), sorted(svs, reverse=True), color='C0', alpha=0.7)
            ax.set_xlabel('SV Index'); ax.set_ylabel('Singular Value')
            ax.set_title(title); ax.set_yscale('log')
        plt.suptitle(f"SV Spectrum Evolution (lr={best_svd_row['lr']}, "
                     f"k={int(best_svd_row['k'])}, B={ctx.bs})", fontsize=14)
        plt.tight_layout()
        plt.savefig(ctx.PLOT_DIR / 'sv_spectrum_evolution.pdf')
        plt.show()

    # Nonzero SVs by k
    best_lr   = best_svd_row['lr']
    best_rtol = best_svd_row['rtol']
    fig, ax = plt.subplots(figsize=(8, 6))
    legend_entries = []
    for i, kf in enumerate(ctx.k_fractions):
        data = ctx.df_svd_fixed_k[
            (ctx.df_svd_fixed_k['lr'] == best_lr) &
            (ctx.df_svd_fixed_k['rtol'] == best_rtol) &
            (ctx.df_svd_fixed_k['k_fraction'] == kf)]
        if len(data) == 0: continue
        row = data.iloc[0]
        if row['svd_info'] is None: continue
        num_nonzero = row['svd_info']['num_nonzero_svs']
        n_ep = len(row['losses']['train'])
        x = np.linspace(0, n_ep, len(num_nonzero))
        sw = max(1, len(num_nonzero) // (n_ep * 5))
        y = sliding_average(num_nonzero, window=sw) if sw > 1 else np.array(num_nonzero)
        xs = x[sw-1:] if sw > 1 else x
        ax.plot(xs, y, f'C{i}-', lw=2)
        legend_entries.append(Line2D([], [], color=f'C{i}',
                                     label=f'k={int(kf*ctx.bs)}'))
    ax.set_xlabel('Epoch'); ax.set_ylabel('# Nonzero Singular Values')
    ax.set_title(f'Nonzero SVs by k (lr={best_lr}, rtol={best_rtol}, B={ctx.bs})')
    if legend_entries: ax.legend(handles=legend_entries)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'nonzero_svs_by_kfraction.pdf')
    plt.show()

### 7. Wall-Time Analysis

In [ ]:
def plot_walltime(ctx):
    """Wall-time bar charts, efficiency scatters, and wall-time vs loss/accuracy curves."""
    best_rows = ctx.best_rows  # set by plot_training_curves

    # Total time for best configs
    fig, ax = plt.subplots(figsize=(8, 5))
    n = len(ctx.best_df); colors = [f'C{i}' for i in range(n)]; x = np.arange(n)
    bars = ax.bar(x, ctx.best_df['total_time'], color=colors)
    ax.set_xticks(x); ax.set_xticklabels(ctx.best_df['optimizer'], rotation=20, ha='right')
    ax.set_ylabel('Total Time (s)')
    ax.set_title(f'Total Training Time (Best Configs, B={ctx.bs})')
    for bar, val in zip(bars, ctx.best_df['total_time']):
        ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.1f}s',
                ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'total_time_best_configs.pdf')
    plt.show()

    # Average batch time
    fig, ax = plt.subplots(figsize=(8, 5))
    batch_times = []; labels = []
    for _, row_data in ctx.best_df.iterrows():
        opt_name = row_data['optimizer']
        if 'SVD' in opt_name:
            if 'fixed' in opt_name:
                r = ctx.df.loc[ctx.df_svd_fixed_k['final_val_loss'].idxmin()]
            else:
                r = ctx.df.loc[ctx.df_svd_var_k['final_val_loss'].idxmin()]
        else:
            opt_df = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt_name]
            r = ctx.df.loc[opt_df['final_val_loss'].idxmin()]
        batch_times.append(r['avg_batch_time_train'])
        labels.append(opt_name)
    x = np.arange(len(labels)); colors = [f'C{i}' for i in range(len(labels))]
    bars = ax.bar(x, [t*1000 for t in batch_times], color=colors)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right')
    ax.set_ylabel('Avg Batch Time (ms)')
    ax.set_title(f'Average Training Batch Time (Best Configs, B={ctx.bs})')
    for bar, val in zip(bars, batch_times):
        ax.text(bar.get_x() + bar.get_width()/2, val*1000, f'{val*1000:.1f}ms',
                ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'avg_batch_time_best_configs.pdf')
    plt.show()

    # Per-epoch time over training
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, (name, row, label) in enumerate(best_rows):
        et = row['losses']['epoch_times']
        ax.plot(range(1, len(et)+1), et, f'C{i}o-', label=label, markersize=4)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Epoch Time (s)')
    ax.set_title(f'Per-Epoch Wall Time (Best Configs, B={ctx.bs})')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'epoch_time_best_configs.pdf')
    plt.show()

    # Loss and accuracy vs wall time
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for i, (name, row, label) in enumerate(best_rows):
        et = row['losses']['epoch_times']
        cum_time = np.concatenate([[0], np.cumsum(et)])
        val_losses = get_loss_curve(row, 'val')
        axes[0].plot(cum_time, val_losses, f'C{i}-', label=label)
        val_acc = get_acc_curve(row, 'val_acc')
        if len(val_acc) > 0:
            axes[1].plot(np.cumsum(et)[:len(val_acc)], val_acc*100, f'C{i}-', label=label)
    axes[0].set_xlabel('Wall Time (s)'); axes[0].set_ylabel('Validation Loss')
    axes[0].set_yscale('log'); axes[0].set_title(f'Val Loss vs Wall Time (B={ctx.bs})')
    axes[0].legend(fontsize=10)
    axes[1].set_xlabel('Wall Time (s)'); axes[1].set_ylabel('Validation Accuracy (%)')
    axes[1].set_title(f'Val Accuracy vs Wall Time (B={ctx.bs})')
    axes[1].legend(fontsize=10, loc='lower right')
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'loss_acc_vs_walltime_best.pdf')
    plt.show()

    # SVD wall time vs k_fraction
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, (subset, title) in zip(axes, [
        (ctx.df_svd_fixed_k, 'Fixed k'), (ctx.df_svd_var_k, 'Variable k')
    ]):
        if len(subset) == 0: continue
        for i, lr in enumerate(ctx.svd_lrs):
            data = subset[subset['lr'] == lr]
            avg_per_kf = data.groupby('k_fraction')['total_time'].mean().reset_index()
            ax.plot(avg_per_kf['k_fraction'] * ctx.bs, avg_per_kf['total_time'], 'o-',
                    label=f'lr={lr}', color=f'C{i}')
        ax.set_xlabel('k'); ax.set_ylabel('Total Training Time (s)')
        ax.set_title(f'SVD Wall Time vs k ({title}, B={ctx.bs})')
        ax.legend()
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'svd_walltime_vs_kfraction.pdf')
    plt.show()

    # Efficiency scatter: val loss vs total time
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, (ycol, ylabel, title) in zip(axes, [
        ('final_val_loss', 'Final Validation Loss', 'Loss vs Wall Time'),
        ('final_val_acc',  'Final Val Accuracy (%)', 'Accuracy vs Wall Time'),
    ]):
        yscale = 100 if 'acc' in ycol else 1
        ax.scatter(ctx.df_svd_fixed_k['total_time'], ctx.df_svd_fixed_k[ycol]*yscale,
                   c='C0', alpha=0.5, label='SVD fixed-k', s=40)
        if len(ctx.df_svd_var_k) > 0:
            ax.scatter(ctx.df_svd_var_k['total_time'], ctx.df_svd_var_k[ycol]*yscale,
                       c='C1', alpha=0.5, label='SVD var-k', s=40, marker='^')
        for i, opt in enumerate(ctx.baseline_optimizers):
            od = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt]
            ax.scatter(od['total_time'], od[ycol]*yscale,
                       c=f'C{i+2}', alpha=0.6, label=opt, s=40, marker='s')
        ax.set_xlabel('Total Training Time (s)'); ax.set_ylabel(ylabel)
        if 'loss' in ycol: ax.set_yscale('log')
        ax.set_title(f'{title} (B={ctx.bs})'); ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'efficiency_scatter.pdf')
    plt.show()

    # ── Excluding first epoch ──────────────────────────────────────────────
    # Compilation overhead report
    if len(ctx.df_svd_fixed_k) > 0:
        best_svd_row = ctx.df.loc[ctx.df_svd_fixed_k['final_val_loss'].idxmin()]
        et = best_svd_row['losses']['epoch_times']
        bt = best_svd_row['losses']['batch_times_train']
        if len(et) > 1 and len(bt) > 1:
            print(f"\nBest SVD (fixed k, B={ctx.bs}) compilation overhead:")
            print(f"  Epoch 1: {et[0]:.2f}s  vs  Epoch 2+ avg: {np.mean(et[1:]):.2f}s "
                  f"({et[0]/np.mean(et[1:]):.1f}x)")
            print(f"  Batch 1: {bt[0]*1000:.1f}ms  vs  Batch 2+ avg: {np.mean(bt[1:])*1000:.2f}ms "
                  f"({bt[0]/np.mean(bt[1:]):.0f}x)")

    # Total time excl. first epoch
    fig, ax = plt.subplots(figsize=(8, 5))
    best_times_excl = []; best_labels = []
    for row_data_opt, subset_key in [('SVD (fixed k)', 'fixed'), ('SVD (var k)', 'var')]:
        subset = ctx.df_svd_fixed_k if subset_key == 'fixed' else ctx.df_svd_var_k
        if len(subset) == 0: continue
        r = ctx.df.loc[subset['final_val_loss'].idxmin()]
        best_times_excl.append(r['time_excl_first_epoch'])
        best_labels.append(row_data_opt)
    for opt in ctx.baseline_optimizers:
        opt_df = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt]
        best_times_excl.append(ctx.df.loc[opt_df['final_val_loss'].idxmin(), 'time_excl_first_epoch'])
        best_labels.append(opt)
    n = len(best_labels); colors = [f'C{i}' for i in range(n)]; x = np.arange(n)
    bars = ax.bar(x, best_times_excl, color=colors)
    ax.set_xticks(x); ax.set_xticklabels(best_labels, rotation=20, ha='right')
    ax.set_ylabel('Total Time (s)')
    ax.set_title(f'Total Training Time Excl. First Epoch (Best Configs, B={ctx.bs})')
    for bar, val in zip(bars, best_times_excl):
        ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.1f}s',
                ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'total_time_excl_first_epoch.pdf')
    plt.show()

    # Avg batch time excl. first batch
    fig, ax = plt.subplots(figsize=(8, 5))
    batch_times_excl = []; labels_excl = []
    for _, row_data in ctx.best_df.iterrows():
        opt_name = row_data['optimizer']
        if 'SVD' in opt_name:
            if 'fixed' in opt_name:
                r = ctx.df.loc[ctx.df_svd_fixed_k['final_val_loss'].idxmin()]
            else:
                r = ctx.df.loc[ctx.df_svd_var_k['final_val_loss'].idxmin()]
        else:
            opt_df = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt_name]
            r = ctx.df.loc[opt_df['final_val_loss'].idxmin()]
        batch_times_excl.append(r['avg_batch_time_excl_first'])
        labels_excl.append(opt_name)
    n = len(labels_excl); colors = [f'C{i}' for i in range(n)]; x = np.arange(n)
    bars = ax.bar(x, [t*1000 for t in batch_times_excl], color=colors)
    ax.set_xticks(x); ax.set_xticklabels(labels_excl, rotation=20, ha='right')
    ax.set_ylabel('Avg Batch Time (ms)')
    ax.set_title(f'Avg Batch Time Excl. First Batch (Best Configs, B={ctx.bs})')
    for bar, val in zip(bars, batch_times_excl):
        ax.text(bar.get_x() + bar.get_width()/2, val*1000, f'{val*1000:.1f}ms',
                ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'avg_batch_time_excl_first_batch.pdf')
    plt.show()

    # Loss/accuracy vs wall time excl. first epoch
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for i, (name, row, label) in enumerate(best_rows):
        et = row['losses']['epoch_times']
        if len(et) < 2: continue
        cum_time_e2 = np.cumsum(et[1:])
        val_losses = get_loss_curve(row, 'val')
        if len(val_losses) > 1:
            axes[0].plot(np.concatenate([[0], cum_time_e2]), val_losses[1:],
                         f'C{i}-', label=label)
        val_acc = get_acc_curve(row, 'val_acc')
        if len(val_acc) > 1:
            axes[1].plot(cum_time_e2[:len(val_acc)-1], val_acc[1:]*100, f'C{i}-', label=label)
    axes[0].set_xlabel('Wall Time (s), excl. first epoch')
    axes[0].set_ylabel('Validation Loss'); axes[0].set_yscale('log')
    axes[0].set_title(f'Val Loss vs Wall Time (excl. 1st epoch, B={ctx.bs})')
    axes[0].legend(fontsize=10)
    axes[1].set_xlabel('Wall Time (s), excl. first epoch')
    axes[1].set_ylabel('Validation Accuracy (%)')
    axes[1].set_title(f'Val Accuracy vs Wall Time (excl. 1st epoch, B={ctx.bs})')
    axes[1].legend(fontsize=10, loc='lower right')
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'loss_acc_vs_walltime_excl_first_epoch.pdf')
    plt.show()

    # Efficiency scatter excl. first epoch
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, (ycol, ylabel, title) in zip(axes, [
        ('final_val_loss', 'Final Validation Loss', 'Loss vs Wall Time (excl. 1st epoch)'),
        ('final_val_acc',  'Final Val Accuracy (%)', 'Accuracy vs Wall Time (excl. 1st epoch)'),
    ]):
        yscale = 100 if 'acc' in ycol else 1
        ax.scatter(ctx.df_svd_fixed_k['time_excl_first_epoch'],
                   ctx.df_svd_fixed_k[ycol]*yscale,
                   c='C0', alpha=0.5, label='SVD fixed-k', s=40)
        if len(ctx.df_svd_var_k) > 0:
            ax.scatter(ctx.df_svd_var_k['time_excl_first_epoch'],
                       ctx.df_svd_var_k[ycol]*yscale,
                       c='C1', alpha=0.5, label='SVD var-k', s=40, marker='^')
        for i, opt in enumerate(ctx.baseline_optimizers):
            od = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt]
            ax.scatter(od['time_excl_first_epoch'], od[ycol]*yscale,
                       c=f'C{i+2}', alpha=0.6, label=opt, s=40, marker='s')
        ax.set_xlabel('Total Time excl. 1st epoch (s)'); ax.set_ylabel(ylabel)
        if 'loss' in ycol: ax.set_yscale('log')
        ax.set_title(f'{title} (B={ctx.bs})'); ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(ctx.PLOT_DIR / 'efficiency_scatter_excl_first_epoch.pdf')
    plt.show()

    # Wall-time vs loss by k value (fixed and variable, incl./excl. first epoch)
    baseline_styles_wt = {
        'Adam': ('C1', 's', 'Adam (best)'),
        'RMSprop': ('C3', 'D', 'RMSprop (best)'),
        'SGD': ('C4', '^', 'SGD (best)'),
    }
    best_baselines = {}
    for opt in ctx.baseline_optimizers:
        opt_data = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt]
        if len(opt_data) > 0:
            best_baselines[opt] = ctx.df_baseline.loc[opt_data['final_val_loss'].idxmin()]

    fixed_k_values = sorted(ctx.df_svd_fixed_k['k'].unique())
    var_k_values   = sorted(ctx.df_svd_var_k['k'].unique()) if len(ctx.df_svd_var_k) > 0 else []
    k_colors_fixed = plt.cm.viridis(np.linspace(0.2, 0.9, max(len(fixed_k_values), 1)))
    k_colors_var   = plt.cm.viridis(np.linspace(0.2, 0.9, max(len(var_k_values), 1)))

    for (k_values, k_data_src, k_colors, ls, fname_stem, title_stem) in [
        (fixed_k_values,  ctx.df_svd_fixed_k, k_colors_fixed, '-',  'fixed_k',    'Fixed-k'),
        (var_k_values,    ctx.df_svd_var_k,   k_colors_var,   '--', 'variable_k', 'Variable-k'),
    ]:
        if len(k_values) == 0: continue
        for excl_first, suffix, xlabel_suffix in [
            (False, 'incl_first', ''),
            (True,  'excl_first', ', excl. first epoch'),
        ]:
            fig, ax = plt.subplots(figsize=(10, 6))
            for i, k in enumerate(k_values):
                kd = k_data_src[k_data_src['k'] == k]
                first = True
                for idx, row in kd.iterrows():
                    et  = row['losses'].get('epoch_times', [])
                    vl  = row['losses'].get('val', [])
                    if not (et and vl): continue
                    if excl_first and len(et) < 2: continue
                    et_use = et[1:] if excl_first else et
                    vl_use = vl[1:] if excl_first else vl
                    ct = np.concatenate([[0], np.cumsum(et_use)])
                    ax.plot(ct, vl_use, color=k_colors[i], alpha=0.6, lw=1.5,
                            linestyle=ls, label=f'k={k}' if first else None)
                    first = False
            for opt_name, (color, marker, label) in baseline_styles_wt.items():
                if opt_name not in best_baselines: continue
                row = best_baselines[opt_name]
                et  = row['losses'].get('epoch_times', [])
                vl  = row['losses'].get('val', [])
                if not (et and vl): continue
                if excl_first and len(et) < 2: continue
                et_use = et[1:] if excl_first else et
                vl_use = vl[1:] if excl_first else vl
                ct = np.concatenate([[0], np.cumsum(et_use)])
                ax.plot(ct, vl_use, color=color, marker=marker, lw=2,
                        markersize=6, markevery=0.15, alpha=0.8, label=label)
            ax.set_xlabel(f'Wall Time (s){xlabel_suffix}')
            ax.set_ylabel('Validation Loss')
            ax.set_title(f'{title_stem}: Wall-Time vs Loss '
                         f'({"Excl." if excl_first else "Incl."} First Epoch, B={ctx.bs})',
                         fontweight='bold')
            ax.legend(loc='best', fontsize=9, ncol=2)
            plt.tight_layout()
            plt.savefig(ctx.PLOT_DIR / f'{fname_stem}_walltime_vs_loss_{suffix}.pdf')
            plt.show()


### 8. Summary

In [ ]:
def print_summary(ctx):
    """Print summary statistics for all optimizer groups."""
    print("=" * 70)
    print(f"SUMMARY STATISTICS  (batch size = {ctx.bs})")
    print("=" * 70)
    print("\nBest performance by optimizer:")
    print(ctx.best_df.to_string(index=False))

    print("\n" + "-" * 70)
    print(f"\nSVD fixed-k ({len(ctx.df_svd_fixed_k)} runs):")
    print(f"  Val loss:  min={ctx.df_svd_fixed_k['final_val_loss'].min():.4f}, "
          f"median={ctx.df_svd_fixed_k['final_val_loss'].median():.4f}")
    print(f"  Val acc:   max={ctx.df_svd_fixed_k['final_val_acc'].max()*100:.1f}%, "
          f"median={ctx.df_svd_fixed_k['final_val_acc'].median()*100:.1f}%")
    print(f"  Time:      min={ctx.df_svd_fixed_k['total_time'].min():.1f}s, "
          f"median={ctx.df_svd_fixed_k['total_time'].median():.1f}s")

    if len(ctx.df_svd_var_k) > 0:
        print(f"\nSVD variable-k ({len(ctx.df_svd_var_k)} runs):")
        print(f"  Val loss:  min={ctx.df_svd_var_k['final_val_loss'].min():.4f}, "
              f"median={ctx.df_svd_var_k['final_val_loss'].median():.4f}")
        print(f"  Val acc:   max={ctx.df_svd_var_k['final_val_acc'].max()*100:.1f}%, "
              f"median={ctx.df_svd_var_k['final_val_acc'].median()*100:.1f}%")
        print(f"  Time:      min={ctx.df_svd_var_k['total_time'].min():.1f}s, "
              f"median={ctx.df_svd_var_k['total_time'].median():.1f}s")

    print(f"\nBaseline optimizers:")
    for opt in ctx.baseline_optimizers:
        od = ctx.df_baseline[ctx.df_baseline['optimizer'] == opt]
        print(f"  {opt} ({len(od)} runs): "
              f"val_loss min={od['final_val_loss'].min():.4f}, "
              f"val_acc max={od['final_val_acc'].max()*100:.1f}%, "
              f"time min={od['total_time'].min():.1f}s")

    if hasattr(ctx, 'shared_configs') and len(ctx.shared_configs) > 0:
        sc = ctx.shared_configs
        print(f"\nVariable k vs Fixed k (matched configs):")
        print(f"  Median loss ratio (var/fixed): {sc['loss_ratio'].median():.3f}")
        print(f"  Median acc diff (var-fixed):   {sc['acc_diff'].median()*100:.2f}%")
        print(f"  Median time ratio (var/fixed): {sc['time_ratio'].median():.3f}")

    print(f"\nAll plots saved to: {ctx.PLOT_DIR.resolve()}")


## Run Analysis for All Batch Sizes

Execute `run_analysis(df, bs)` for each batch size found in the data.
To run for a single batch size, call it directly: `run_analysis(df, 64)`


In [ ]:
def run_analysis(df, bs):
    """Run the full analysis pipeline for one batch size."""
    ctx = build_context(df, bs)
    plot_best_performance(ctx)
    plot_training_curves(ctx)
    plot_variable_k(ctx)
    plot_hyperparam_sensitivity(ctx)
    plot_singular_values(ctx)
    #plot_walltime(ctx)
    #print_summary(ctx)
    return ctx


# Run for every batch size in the data
results = {}
for bs in batch_sizes:
    results[bs] = run_analysis(df, bs)


## Train/Val Loss vs Wall Time

In [ ]:
# Train and val loss vs cumulative wall time for best config of each optimizer
# One subplot per batch size
for bs, ctx in results.items():
    fig, ax = plt.subplots(figsize=(10, 6))

    for i, (name, row, label) in enumerate(ctx.best_rows):
        et = row['losses'].get('epoch_times', [])
        if not et:
            continue
        cum_time = np.concatenate([[0], np.cumsum(et)])

        val_curve = get_loss_curve(row, 'val')   # length n_epochs+1
        train_curve = get_loss_curve(row, 'train')  # length n_epochs

        ax.plot(cum_time, val_curve,   f'C{i}--', lw=2, label=f'{label} (val)')
        ax.plot(cum_time[1:], train_curve, f'C{i}-',  lw=2, label=f'{label} (train)')

    ax.set_xlabel('Wall Time (s)')
    ax.set_ylabel('Loss')
    ax.set_yscale('log')
    ax.set_title(f'Train/Val Loss vs Wall Time (B={bs})')

    # Two-part legend: optimizer colors + linestyle meaning
    handles, labels_leg = ax.get_legend_handles_labels()
    # optimizer entries (every other starting at 0 = val lines)
    opt_handles = handles[::2]
    opt_labels  = [l.rsplit(' (', 1)[0] for l in labels_leg[::2]]
    style_handles = [
        Line2D([], [], color='k', linestyle='-',  lw=2, label='Train'),
        Line2D([], [], color='k', linestyle='--', lw=2, label='Val'),
    ]
    leg1 = ax.legend(handles=opt_handles,   labels=opt_labels,   loc='upper right', fontsize=10)
    ax.add_artist(leg1)
    ax.legend(handles=style_handles, loc='lower left', fontsize=10)

    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'train_val_loss_vs_walltime_bs{bs}.pdf')
    plt.show()